<a href="https://colab.research.google.com/github/sabrinaangel/ml-dataset-preprocessing/blob/main/Week%208-9-10/03_asl_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="./images/DLI_Header.png" style="width: 400px;">

# Convolutional Neural Networks

In the previous section, we built and trained a simple model to classify ASL images. The model was able to learn how to correctly classify the training dataset with very high accuracy, but, it did not perform nearly as well on validation dataset. This behavior of not generalizing well to non-training data is called [overfitting](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html), and in this section, we will introduce a popular kind of model called a [convolutional neural network](https://towardsdatascience.com/a-comprehensive-guide-to-convolutional-neural-networks-the-eli5-way-3bd2b1164a53) that is especially good for reading images and classifying them.

## Objectives

* Prep data specifically for a CNN
* Create a more sophisticated CNN model, understanding a greater variety of model layers
* Train a CNN model and observe its performance

## Loading and Preparing the Data

The below cell contains the data preprocessing techniques we learned in the previous labs. Review it and execute it before moving on:

In [3]:
import tensorflow.keras as keras
import pandas as pd

# Load in our data from CSV files
train_df = pd.read_csv("sign_mnist_train.csv")
valid_df = pd.read_csv("sign_mnist_test.csv")

# Separate out our target values
y_train = train_df['label']
y_valid = valid_df['label']
del train_df['label']
del valid_df['label']

# Adjust labels to account for the missing letter 'J' (label 9 in a 0-25 alphabet)
# By decrementing labels >= 9, we map the original 0-8, 10-24 range to a contiguous 0-23 range.
y_train = y_train.apply(lambda y: y - 1 if y >= 9 else y)
y_valid = y_valid.apply(lambda y: y - 1 if y >= 9 else y)

# Separate out our image vectors
x_train = train_df.values
x_valid = valid_df.values

# Turn our scalar targets into binary categories
num_classes = 24  # Total 24 classes (A-I, K-Y)
y_train = keras.utils.to_categorical(y_train, num_classes)
y_valid = keras.utils.to_categorical(y_valid, num_classes)

# Normalize our image data
x_train = x_train / 255
x_valid = x_valid / 255

## Reshaping Images for a CNN

In the last exercise, the individual pictures in our dataset are in the format of long lists of 784 pixels:

In [4]:
x_train.shape, x_valid.shape

((27455, 784), (7172, 784))

In this format, we don't have all the information about which pixels are near each other. Because of this, we can't apply convolutions that will detect features. Let's reshape our dataset so that they are in a 28x28 pixel format. This will allow our convolutions to associate groups of pixels and detect important features.

Note that for the first convolutional layer of our model, we need to have not only the height and width of the image, but also the number of [color channels](https://www.photoshopessentials.com/essentials/rgb/). Our images are grayscale, so we'll just have 1 channel.

That means that we need to convert the current shape `(27455, 784)` to `(27455, 28, 28, 1)`. As a convenience, we can pass the [reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html#numpy.reshape) method a `-1` for any dimension we wish to remain the same, therefore:

In [5]:
x_train = x_train.reshape(-1,28,28,1)
x_valid = x_valid.reshape(-1,28,28,1)

In [6]:
x_train.shape

(27455, 28, 28, 1)

In [7]:
x_valid.shape

(7172, 28, 28, 1)

In [8]:
x_train.shape, x_valid.shape

((27455, 28, 28, 1), (7172, 28, 28, 1))

## Creating a Convolutional Model

These days, many data scientists start their projects by borrowing model properties from a similar project. Assuming the problem is not totally unique, there's a great chance that people have created models that will perform well which are posted in online repositories like [TensorFlow Hub](https://www.tensorflow.org/hub) and the [NGC Catalog](https://ngc.nvidia.com/catalog/models). Today, we'll provide a model that will work well for this problem.

<img src="images/cnn.png" width=180 />

We covered many of the different kinds of layers in the lecture, and we will go over them all here with links to their documentation. When in doubt, read the official documentation (or ask [stackoverflow](https://stackoverflow.com/)).

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    MaxPool2D,
    Flatten,
    Dropout,
    BatchNormalization,
)

model = Sequential()
model.add(Conv2D(75, (3, 3), strides=1, padding="same", activation="relu",
                 input_shape=(28, 28, 1)))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2), strides=2, padding="same"))
model.add(Conv2D(50, (3, 3), strides=1, padding="same", activation="relu"))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2), strides=2, padding="same"))
model.add(Conv2D(25, (3, 3), strides=1, padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2), strides=2, padding="same"))
model.add(Flatten())
model.add(Dense(units=512, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(units=num_classes, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)

<img src="images/conv2d.png" width=300 />

These are our 2D convolutional layers. Small kernels will go over the input image and detect features that are important for classification. Earlier convolutions in the model will detect simple features such as lines. Later convolutions will detect more complex features. Let's look at our first Conv2D layer:
```Python
model.add(Conv2D(75 , (3,3) , strides = 1 , padding = 'same'...)
```
75 refers to the number of filters that will be learned. (3,3) refers to the size of those filters. Strides refer to the step size that the filter will take as it passes over the image. Padding refers to whether the output image that's created from the filter will match the size of the input image.

### [BatchNormalization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization)

Like normalizing our inputs, batch normalization scales the values in the hidden layers to improve training. [Read more about it in detail here](https://blog.paperspace.com/busting-the-myths-about-batch-normalization/).

### [MaxPool2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D)

<img src="images/maxpool2d.png" width=300 />
Max pooling takes an image and essentially shrinks it to a lower resolution. It does this to help the model be robust to translation (objects moving side to side), and also makes our model faster.

### [Dropout](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout)

<img src="images/dropout.png" width=360 />
Dropout is a technique for preventing overfitting. Dropout randomly selects a subset of neurons and turns them off, so that they do not participate in forward or backward propagation in that particular pass. This helps to make sure that the network is robust and redundant, and does not rely on any one area to come up with answers.    

### [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)

Flatten takes the output of one layer which is multidimensional, and flattens it into a one-dimensional array. The output is called a feature vector and will be connected to the final classification layer.

### [Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)

We have seen dense layers before in our earlier models. Our first dense layer (512 units) takes the feature vector as input and learns which features will contribute to a particular classification. The second dense layer (24 units) is the final classification layer that outputs our prediction.

## Summarizing the Model

This may feel like a lot of information, but don't worry. It's not critical that to understand everything right now in order to effectively train convolutional models. Most importantly we know that they can help with extracting useful information from images, and can be used in classification tasks.

Here, we summarize the model we just created. Notice how it has fewer trainable parameters than the model in the previous notebook:

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 75)     │           750 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 75)     │           300 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 75)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 50)     │        33,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 50)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 50)     │           200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 25)       │        11,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 25)       │           100 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 25)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       205,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │        12,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 264,049 (1.01 MB)

 Trainable params: 263,749 (1.01 MB)

 Non-trainable params: 300 (1.17 KB)

## Compiling the Model

We'll compile the model just like before:

In [11]:
model.compile(loss="categorical_crossentropy", metrics=["accuracy"])

## Training the Model

Despite the very different model architecture, the training looks exactly the same. Run the cell below to train for 20 epochs and let's see if the accuracy improves:

In [12]:
history = model.fit(x_train, y_train, epochs=20, verbose=1, validation_data=(x_valid, y_valid))

Epoch 1/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 103s 117ms/step - accuracy: 0.9032 - loss: 0.3123 - val_accuracy: 0.8826 - val_loss: 0.3753
Epoch 2/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 140s 115ms/step - accuracy: 0.9940 - loss: 0.0190 - val_accuracy: 0.9519 - val_loss: 0.1897
Epoch 3/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 100s 116ms/step - accuracy: 0.9962 - loss: 0.0118 - val_accuracy: 0.9225 - val_loss: 0.2696
Epoch 4/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 100s 117ms/step - accuracy: 0.9978 - loss: 0.0067 - val_accuracy: 0.9690 - val_loss: 0.0875
Epoch 5/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 147s 123ms/step - accuracy: 0.9980 - loss: 0.0057 - val_accuracy: 0.9169 - val_loss: 0.3013
Epoch 6/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 101s 118ms/step - accuracy: 0.9992 - loss: 0.0027 - val_accuracy: 0.9481 - val_loss: 0.1990
Epoch 7/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 99s 115ms/step - accuracy: 0.9995 - loss: 0.0015 - val_accuracy: 0.9642 - val_loss: 0.1409
Epoch 8/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 143s 116ms/step - accuracy: 0.9993 - 

## Menyimpan dan Memuat Model Keras

Setelah model dilatih, sangat penting untuk menyimpannya agar dapat digunakan kembali tanpa perlu melatih ulang. Keras menyediakan cara mudah untuk menyimpan seluruh model (arsitektur, bobot, konfigurasi *optimizer*, dan *state* pelatihan) ke dalam satu file.

In [2]:
# Menyimpan model
model.save('my_sign_language_model.keras')
print('Model berhasil disimpan sebagai my_sign_language_model.keras')

NameError: name 'model' is not defined

Anda dapat memuat model yang telah disimpan kapan saja, bahkan di sesi Python yang berbeda, untuk melakukan prediksi atau melanjutkan pelatihan.

In [3]:
from tensorflow.keras.models import load_model

# Memuat model yang telah disimpan
loaded_model = load_model('my_sign_language_model.keras')
print('Model berhasil dimuat!')

# Anda bisa mengecek ringkasan model yang dimuat
loaded_model.summary()

ValueError: File not found: filepath=my_sign_language_model.keras. Please ensure the file is an accessible `.keras` zip file.

Sekarang Anda dapat menggunakan `loaded_model` untuk membuat prediksi atau mengevaluasi kinerja pada data baru, sama seperti model aslinya.

## Discussion of Results

It looks like this model is significantly improved! The training accuracy is very high, and the validation accuracy has improved as well. This is a great result, as all we had to do was swap in a new model.

You may have noticed the validation accuracy jumping around. This is an indication that our model is still not generalizing perfectly. Fortunately, there's more that we can do. Let's talk about it in the next lecture.

## Summary

In this section, we utilized several new kinds of layers to implement a CNN, which performed better than the more simple model used in the last section. Hopefully the overall process of creating and training a model with prepared data is starting to become even more familiar.

## Clear the Memory
Before moving on, please execute the following cell to clear up the GPU memory. This is required to move on to the next notebook.

In [1]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

## Next

In the last several sections you have focused on the creation and training of models. In order to further improve performance, you will now turn your attention to *data augmentation*, a collection of techniques that will allow your models to train on more and better data than what you might have originally at your disposal.

## Uji Coba dan Refleksi Pribadi (Evaluasi Model Tersimpan)

Setelah model dilatih dan disimpan, kita tidak perlu melatih ulang setiap kali ingin mengecek performanya. Di bagian ini, kita akan memuat kembali model yang telah disimpan (`my_sign_language_model.keras`) dan mengevaluasinya pada dataset validasi untuk melihat seberapa baik model tersebut menggeneralisasi data yang belum pernah dilihat sebelumnya.

In [4]:
from tensorflow.keras.models import load_model

# Pastikan model sudah tersimpan dan ada di direktori yang sama
# Jika Anda sudah menjalankan sel 'Memuat model yang telah disimpan' sebelumnya
# maka 'loaded_model' sudah tersedia.
# Jika belum, kita muat ulang di sini.
try:
    if 'loaded_model' not in locals():
        loaded_model = load_model('my_sign_language_model.keras')
        print("Model berhasil dimuat untuk evaluasi.")
except Exception as e:
    print(f"Gagal memuat model: {e}. Pastikan model telah disimpan terlebih dahulu.")
    # Optionally, you could try to re-save and load here if the original model is still in memory
    # if 'model' in locals():
    #    model.save('my_sign_language_model.keras')
    #    loaded_model = load_model('my_sign_language_model.keras')
    #    print("Model 'model' yang aktif telah disimpan dan dimuat ulang.")
    # else:
    #    print("Model 'model' juga tidak ditemukan di memori.")


# Evaluasi model yang dimuat pada data validasi
if 'loaded_model' in locals():
    print("\nMengevaluasi performa model pada data validasi...")
    loss, accuracy = loaded_model.evaluate(x_valid, y_valid, verbose=1)
    print(f"\nLoss Validasi: {loss:.4f}")
    print(f"Akurasi Validasi: {accuracy:.4f}")
else:
    print("Tidak dapat melakukan evaluasi karena model belum berhasil dimuat.")

Gagal memuat model: File not found: filepath=my_sign_language_model.keras. Please ensure the file is an accessible `.keras` zip file.. Pastikan model telah disimpan terlebih dahulu.
Tidak dapat melakukan evaluasi karena model belum berhasil dimuat.


### Kesimpulan dari Pengamatan (Evaluasi Model Tersimpan)

Dari hasil evaluasi model yang sudah dimuat, kita bisa mendapatkan gambaran langsung tentang performa model pada data validasi. Angka akurasi validasi menunjukkan seberapa baik model dapat menggeneralisasi pada data yang belum pernah dilihat sebelumnya. Jika akurasi validasi cukup tinggi, ini menandakan model bekerja dengan baik.

Jika Anda melihat ada perbedaan signifikan antara akurasi training (dari output `model.fit()` terakhir) dengan akurasi validasi ini, itu bisa menjadi indikasi *overfitting*. Namun, pendekatan evaluasi ini memungkinkan kita untuk cepat memeriksa performa model tanpa harus menjalankan proses pelatihan yang lama setiap saat.